# ⚡ Arbitrage Calculator: Cloud GPU ML Matcher (v2.0)
Pulls markets via ngrok, sends matches back via ngrok. Backup file saved for retries.

### Instructions
1. Go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.
2. Click **Runtime > Run all**.
3. The matched arbitrage pairs will be posted back to your local dashboard.

In [ ]:
# 1. Install required packages
!pip install sentence-transformers torch httpx pydantic -q

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import httpx
import asyncio
from typing import List, Dict, Any
import time

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Load ML Models directly onto the Cloud GPU
print("Loading Bi-Encoder (Semantic Matrix)...")
bi_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda' if torch.cuda.is_available() else 'cpu')

print("Loading Cross-Encoder (Nuance/Reasoning Engine)...")
cross_model = CrossEncoder('cross-encoder/nli-deberta-v3-small', device='cuda' if torch.cuda.is_available() else 'cpu')
print("Models loaded successfully!")

In [ ]:
# 3. Configure Local Connection
LOCAL_BACKEND_URL = "https://copyrightable-pseudocartilaginous-sade.ngrok-free.dev/api"

def fetch_markets_from_local():
    import httpx
    import json
    import os
    # FALLBACK: Load from backup file if ngrok fails
    if os.path.exists('/tmp/markets_backup.json'):
        print("Loading markets from backup file (retry case)...")
        with open('/tmp/markets_backup.json', 'r') as f:
            return json.load(f)
    print(f"Fetching markets from local backend via ngrok: {LOCAL_BACKEND_URL}/raw-markets...")
    response = httpx.get(f"{LOCAL_BACKEND_URL}/raw-markets", timeout=120.0)
    response.raise_for_status()
    markets = response.json()
    # Save backup locally for future retries
    with open('/tmp/markets_backup.json', 'w') as f:
        json.dump(markets, f)
    return markets

all_markets = fetch_markets_from_local()
print(f"Fetched {len(all_markets)} markets for processing.")

In [ ]:
def get_key_numbers(text):
    import re
    parsed = set()
    for n in re.findall(r'\d+(?:\.\d+)?', text):
        val = float(n)
        if val not in (2024, 2025, 2026, 2027):
            parsed.add(val)
    return parsed

# 4. Cloud GPU Matrix Multiplication Logic with CROSS-ENCODER (Nuance Detection)
def compute_pair_arb(ma, mb):
    price1_a, price1_b = ma['yesPrice'], 1 - mb['yesPrice']
    price2_a, price2_b = 1 - ma['yesPrice'], mb['yesPrice']
    
    cost1 = price1_a + price1_b
    roi1 = ((1 - cost1) / cost1 * 100) if cost1 < 1 and cost1 > 0 else -100
    
    cost2 = price2_a + price2_b
    roi2 = ((1 - cost2) / cost2 * 100) if cost2 < 1 and cost2 > 0 else -100
    
    if roi1 > roi2:
        return {"roi": roi1, "cost": cost1, "scenario": 1}
    return {"roi": roi2, "cost": cost2, "scenario": 2}

def match_on_gpu(markets, min_similarity=65.0, min_roi=0.1, top_k=2000):
    by_platform = {}
    for m in markets:
        if m.get("isBinary", True) and m.get("outcomeCount", 2) == 2:
            by_platform.setdefault(m["platform"], []).append(m)
            
    platforms = list(by_platform.keys())
    print(f"Processing platforms: {platforms}")
    
    # 1. Generate Embeddings on GPU (Bi-Encoder Phase)
    start_time = time.time()
    plat_embeddings = {}
    for plat in platforms:
        titles = [m["title"] for m in by_platform[plat]]
        print(f"Encoding {len(titles)} markets for {plat}...")
        plat_embeddings[plat] = bi_model.encode(titles, convert_to_tensor=True, batch_size=256)
        
    # 2. Candidate Search (Fast Matrix Mul)
    candidates = []
    threshold_tensor = min_similarity / 100.0
    
    print("\nExecuting Bi-Encoder Matrix Multiplications (Fast Filters)...")
    for i in range(len(platforms)):
        pa = platforms[i]
        emb_a = plat_embeddings[pa]
        if emb_a is None: continue
        
        for j in range(i + 1, len(platforms)):
            pb = platforms[j]
            emb_b = plat_embeddings[pb]
            if emb_b is None: continue
            
            cosine_scores = util.cos_sim(emb_a, emb_b)
            high_score_indices = (cosine_scores >= threshold_tensor).nonzero(as_tuple=False)
            
            for idx in high_score_indices:
                idx_a, idx_b = idx[0].item(), idx[1].item()
                ma, mb = by_platform[pa][idx_a], by_platform[pb][idx_b]
                
                # --- NUMERICAL CONFLICT FILTER ---
                nums_a = get_key_numbers(ma['title'])
                nums_b = get_key_numbers(mb['title'])
                if nums_a and nums_b and nums_a.isdisjoint(nums_b):
                    continue

                arb_data = compute_pair_arb(ma, mb)
                if arb_data["roi"] >= min_roi:
                    candidates.append((ma, mb, arb_data["roi"]))
    
    # 3. Nuance Reranking (Cross-Encoder Phase)
    if not candidates:
        return []
        
    print(f"\nApplying Nuance Reranking (Cross-Encoder) to top {len(candidates)} candidates...")
    candidates.sort(key=lambda x: x[2], reverse=True)
    candidates_to_rerank = candidates[:2000]
    
    pairs_to_score = [[c[0]['title'], c[1]['title']] for c in candidates_to_rerank]
    
    cross_scores = cross_model.predict(pairs_to_score, show_progress_bar=True)
    
    final_pairs = []
    for i, score in enumerate(cross_scores):
        ma, mb, roi = candidates_to_rerank[i]
        # NLI model returns [contradiction, neutral, entailment] - get entailment (index 2)
        if hasattr(score, '__len__') and len(score) > 1:
            score_val = float(score[2])  # entailment score
        else:
            score_val = float(score)
        final_pairs.append({
            "marketA": ma, 
            "marketB": mb, 
            "roi": roi,
            "matchScore": min(100.0, max(0.0, score_val * 10.0)),
            "isVerified": True if score_val > 5.0 else False
        })
    
    final_pairs.sort(key=lambda x: (x["isVerified"], x["roi"]), reverse=True)
    final_pairs = final_pairs[:top_k]
    
    end_time = time.time()
    print(f"\nML Nuance Discovery Complete! Verified {len(final_pairs)} matches in {end_time - start_time:.2f} seconds.")
    return final_pairs


In [ ]:
if all_markets:
    found_pairs = match_on_gpu(all_markets, min_similarity=55.0, min_roi=0.1, top_k=2000)
else:
    print("No markets loaded to process.")
    found_pairs = []

# 5. Send matched pairs back to local backend for LLM analysis
if found_pairs:
    print(f"\nSending {len(found_pairs)} matched pairs to local backend...")
    try:
        response = httpx.post(
            f"{LOCAL_BACKEND_URL}/cloud-results",
            json=found_pairs,
            timeout=60.0
        )
        response.raise_for_status()
        print(f"✓ Sent {len(found_pairs)} pairs to local backend!")
    except Exception as e:
        print(f"✗ Failed to send results: {e}")
else:
    print("No arbitrage pairs found.")